# Colab Local Storage Setup (No Google Drive dataset upload)

This notebook shows how to:
- store a large dataset in Colab local storage (`/content`)
- check free space before extraction
- extract archives safely
- train from local disk
- save only checkpoints/logs to persistent storage

In [ ]:
# 1) Paths and basic checks
import shutil
from pathlib import Path

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Dataset archive path (example)
ARCHIVE_PATH = DATA_DIR / 'dataset.zip'

# Extract destination
EXTRACT_DIR = DATA_DIR / 'images'
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

def gb(x):
    return x / (1024 ** 3)

total, used, free = shutil.disk_usage('/content')
print(f'Disk total: {gb(total):.1f} GB')
print(f'Disk used : {gb(used):.1f} GB')
print(f'Disk free : {gb(free):.1f} GB')

In [ ]:
# 2) Download dataset directly to /content
# Option A: direct URL
# !wget -O /content/data/dataset.zip "https://your-url/dataset.zip"

# Option B: Google Cloud Storage
# !gsutil -m cp "gs://your-bucket/dataset.zip" /content/data/dataset.zip

# Option C: Kaggle
# !pip -q install kaggle
# !kaggle datasets download -d owner/dataset-name -p /content/data

print('Archive exists:', ARCHIVE_PATH.exists())
if ARCHIVE_PATH.exists():
    print('Archive size GB:', ARCHIVE_PATH.stat().st_size / (1024 ** 3))

In [ ]:
# 3) Safety check before extraction
# Heuristic: free space >= 2x archive size
if ARCHIVE_PATH.exists():
    archive_size = ARCHIVE_PATH.stat().st_size
    _, _, free = shutil.disk_usage('/content')
    ratio = free / archive_size
    print(f'Free/archive ratio: {ratio:.2f}x')
    if ratio < 2.0:
        print('WARNING: Low headroom. Extraction/training may run out of space.')
else:
    print('No archive found yet. Download it first.')

In [ ]:
# 4) Extract into /content
# Run the command that matches your archive format.

# ZIP:
# !unzip -q /content/data/dataset.zip -d /content/data/images

# TAR.GZ:
# !tar -xzf /content/data/dataset.tar.gz -C /content/data/images

# 7z:
# !apt-get -qq install -y p7zip-full
# !7z x /content/data/dataset.7z -o/content/data/images -y

print('Choose one extraction command above and run it.')

In [ ]:
# 5) Verify dataset and reclaim some space
!find /content/data/images -type f | wc -l

# Optional: delete archive after successful extraction
# !rm -f /content/data/dataset.zip

In [ ]:
# 6) Save checkpoints to persistent storage (small data only)
from google.colab import drive

drive.mount('/content/drive')

CKPT_DIR = Path('/content/drive/MyDrive/oilspill_checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoint dir:', CKPT_DIR)

# Save checkpoints here during training.
# Example (PyTorch):
# torch.save(model.state_dict(), CKPT_DIR / f'epoch_{epoch:03d}.pt')

In [ ]:
# 7) Quick free-space monitor during training
import shutil
_, _, free = shutil.disk_usage('/content')
print(f'Free disk: {free/(1024**3):.1f} GB')